## this first saves .viser grasp animatio then , visualizes all grasps at time with pcd

In [8]:
import numpy as np
import open3d as o3d
import os
from makegripper_points import plot_gripper_pro_max
from pathlib import Path
import time
import viser

# --- Setup and Data Loading ---
server = viser.ViserServer()
data_dir = './antipodal/blue_cylinder/inputs/'
cloud = o3d.io.read_point_cloud(os.path.join(data_dir, "cloud.ply"))
grasp_data = np.load(os.path.join(data_dir, "grasps.npy"), allow_pickle=True).item()

# --- List to hold geometry for the final Open3D visualization ---
geometries_for_open3d = []

# --- NEW: Crop Point Cloud FIRST ---
print("Cropping point cloud where x < 0.6...")
points = np.asarray(cloud.points)
# Find the indices of points that satisfy the condition
crop_indices = np.where(points[:, 0] < 0.6)[0]
# Create a new point cloud object containing only the selected points
cropped_pcd = cloud.select_by_index(crop_indices)
print(f"  - Original points: {len(cloud.points)}, Cropped points: {len(cropped_pcd.points)}")


# --- MODIFIED: Plane Segmentation and Coloring on CROPPED cloud ---
print("Detecting plane on cropped point cloud...")
if not cropped_pcd.has_colors():
    cropped_pcd.paint_uniform_color([0.5, 0.5, 0.5])

# Run plane detection on the smaller, cropped point cloud
plane_model, inliers = cropped_pcd.segment_plane(
    distance_threshold=0.005,
    ransac_n=3,
    num_iterations=1000
)
print(f"  - Found {len(inliers)} points belonging to the plane.")
# Get the colors from the cropped cloud and modify them
final_colors = np.asarray(cropped_pcd.colors)
final_colors[inliers] = [0.9, 0.8, 0.9] 


# --- MODIFIED: Create a new Open3D PointCloud object for the preview ---
# This object is now built from the cropped and colored data
o3d_colored_cloud = o3d.geometry.PointCloud()
o3d_colored_cloud.points = cropped_pcd.points
o3d_colored_cloud.colors = o3d.utility.Vector3dVector(final_colors)
geometries_for_open3d.append(o3d_colored_cloud)
geometries_for_open3d.append(o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.1))


# --- MODIFIED: Add Static Scene Elements to Viser ---
# Send the cropped and colored point cloud to Viser
server.add_point_cloud(
    name="/scene/point_cloud",
    points=np.asarray(cropped_pcd.points),
    colors=final_colors,
    point_size=0.002
)

# --- Create Grasp Geometry for Viser (Initially Hidden) ---
# This loop populates both the Viser scene and the Open3D preview list
grasp_indices = [11, 2, 4, 9, 3, 0, 14, 16, 5, 12]
all_grasp_handles = []
for idx in grasp_indices:
    t = grasp_data['translations'][idx]
    R_mat = grasp_data['rotations'][idx]
    
    gripper_mesh, _ = plot_gripper_pro_max(t, R_mat, width=grasp_data['widths'][idx], depth=grasp_data['heights'][idx])
    
    # Add the mesh to our Open3D list for the final preview
    geometries_for_open3d.append(gripper_mesh)
    
    # Add the mesh to the Viser scene but keep it hidden
    handle = server.add_mesh(
        name=f"/grasps/grasp_{idx}",
        vertices=np.asarray(gripper_mesh.vertices),
        faces=np.asarray(gripper_mesh.triangles),
        color=(0.0, 1.0, 0.0),
        visible=False,
    )
    all_grasp_handles.append(handle)

# --- Dynamic Scene Export (Animation) FIRST ---
print("\nRecording Viser animation...")
recorder = server._start_scene_recording()
recorder.set_loop_start()

for i, handle in enumerate(all_grasp_handles):
    print(f"  - Frame {i+1}/{len(all_grasp_handles)}")
    handle.visible = True
    recorder.insert_sleep(1.0)
    handle.visible = False

# Save the Final Animation
output_filename = "grasp_animation_colored_plane.viser"
print(f"Saving animation to {output_filename}...")
Path(output_filename).write_bytes(recorder.end_and_serialize())
print("File saved successfully.")

# --- Show the Open3D Preview Window LAST ---
print("\n.viser file saved. Now showing static preview in Open3D window.")
print("Close the Open3D window to exit the script.")
o3d.visualization.draw_geometries(
    geometries_for_open3d,
    window_name="Static Preview (after saving .viser)"
)

print("Script finished.")

╭──────────────── viser ────────────────╮
│             ╷                         │
│   HTTP      │ http://localhost:8100   │
│   Websocket │ ws://localhost:8100     │
│             ╵                         │
╰───────────────────────────────────────╯

Cropping point cloud where x < 0.6...
  - Original points: 80221, Cropped points: 74122
Detecting plane on cropped point cloud...
  - Found 69355 points belonging to the plane.

Recording Viser animation...
  - Frame 1/10
  - Frame 2/10
  - Frame 3/10
  - Frame 4/10
  - Frame 5/10
  - Frame 6/10
  - Frame 7/10
  - Frame 8/10
  - Frame 9/10
  - Frame 10/10
Saving animation to grasp_animation_colored_plane.viser...
File saved successfully.

.viser file saved. Now showing static preview in Open3D window.
Close the Open3D window to exit the script.


/tmp/ipykernel_7025/4266686465.py:56: DeprecationWarning: ViserServer.add_point_cloud has been deprecated, use ViserServer.scene.add_point_cloud instead. Alternatively, pin to `viser<0.2.0`.
  server.add_point_cloud(
/tmp/ipykernel_7025/4266686465.py:77: DeprecationWarning: ViserServer.add_mesh has been deprecated, use ViserServer.scene.add_mesh_simple instead. Alternatively, pin to `viser<0.2.0`.
  handle = server.add_mesh(


Script finished.
